In [0]:
from pyspark.sql.functions import col, lit
import datetime, json

# Table paths
silver_path = "abfss://silver@strdatabrickssadls.dfs.core.windows.net/promotion"
silver_table = "db_dataclassdev.silver.promotion"

# Read bronze table
df_bronze = spark.read.format("delta").table("db_dataclassdev.bronze.promotion")

# Cast columns to required types
df_transformed = (
    df_bronze
    .withColumn("code", col("code").cast("long"))
    .withColumn("supermarkets", col("supermarkets").cast("int"))
    .withColumn("week", col("week").cast("int"))
    .withColumn("province", col("province").cast("int"))
    .withColumn("etl_record_created_date", lit(datetime.datetime.utcnow()))
    .withColumn("etl_record_modified_date", lit(datetime.datetime.utcnow()))
)

# Create silver table if not exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_table} (
    code BIGINT,
    supermarkets INT,
    week INT,
    feature STRING,
    display STRING,
    province INT,
    etl_record_created_date TIMESTAMP,
    etl_record_modified_date TIMESTAMP
)
USING DELTA
LOCATION '{silver_path}'
""")

# Merge logic
from delta.tables import DeltaTable

target = DeltaTable.forPath(spark, silver_path)

(
    target.alias("target")
    .merge(
        df_transformed.alias("source"),
        """
        target.code = source.code AND 
        target.supermarkets = source.supermarkets AND 
        target.week = source.week
        """
    )
    .whenMatchedUpdate(set={
        "feature": "source.feature",
        "display": "source.display",
        "province": "source.province",
        "etl_record_modified_date": "source.etl_record_modified_date"
    })
    .whenNotMatchedInsertAll()
    .execute()
)

# Prepare audit info
audit_info = {
    "fileName": "promotion",
    "rowCount": df_transformed.count(),
    "status": "Succeeded",
    "destinationPath": silver_path,
    "timestamp": str(datetime.datetime.utcnow())
}

dbutils.notebook.exit(json.dumps(audit_info))
